# TMDB Movie Recommender

This notebook loads the TMDB dataset, prepares a simple metadata-based recommendation model, and saves the model files used by the app.

In [1]:
from pathlib import Path
import pickle

import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


## 1) Load data

In [3]:
project_root = Path.cwd()
data_candidates = [
    project_root / 'src' / 'data' / 'top10K-TMDB-movies.csv',
    project_root.parent / 'data' / 'top10K-TMDB-movies.csv',
]
data_path = next((path for path in data_candidates if path.exists()), None)
assert data_path is not None, 'Dataset not found in src/data'
model_dir = data_path.parent.parent / 'models'
model_dir.mkdir(parents=True, exist_ok=True)

movies_df = pd.read_csv(data_path)
movies_df.head()

,id,title,genre,original_language,overview,popularity,release_date,vote_average,vote_count
0,278,The Shawshank Redemption,"Drama,Crime",en,Framed in the 1940s for the double murder of h...,94.075,1994-09-23,8.7,21862
1,19404,Dilwale Dulhania Le Jayenge,"Comedy,Drama,Romance",hi,"Raj is a rich, carefree, happy-go-lucky second...",25.408,1995-10-19,8.7,3731
2,238,The Godfather,"Drama,Crime",en,"Spanning the years 1945 to 1955, a chronicle o...",90.585,1972-03-14,8.7,16280
3,424,Schindler's List,"Drama,History,War",en,The true story of how businessman Oskar Schind...,44.761,1993-12-15,8.6,12959
4,240,The Godfather: Part II,"Drama,Crime",en,In the continuing saga of the Corleone crime f...,57.749,1974-12-20,8.6,9811


In [ ]:
movies_df.shape
movies_df.isnull().sum()

id                    0
title                 0
genre                 3
original_language     0
overview             13
popularity            0
release_date          0
vote_average          0
vote_count            0
dtype: int64

In [ ]:
selected_columns = ['id', 'title', 'genre', 'original_language', 'overview', 'popularity', 'release_date', 'vote_average', 'vote_count']
movies_df = movies_df[selected_columns].copy()
movies_df['genre'] = movies_df['genre'].fillna('')
movies_df['overview'] = movies_df['overview'].fillna('')
movies_df.head()

,id,title,genre,original_language,overview,popularity,release_date,vote_average,vote_count
0,278,The Shawshank Redemption,"Drama,Crime",en,Framed in the 1940s for the double murder of h...,94.075,1994-09-23,8.7,21862
1,19404,Dilwale Dulhania Le Jayenge,"Comedy,Drama,Romance",hi,"Raj is a rich, carefree, happy-go-lucky second...",25.408,1995-10-19,8.7,3731
2,238,The Godfather,"Drama,Crime",en,"Spanning the years 1945 to 1955, a chronicle o...",90.585,1972-03-14,8.7,16280
3,424,Schindler's List,"Drama,History,War",en,The true story of how businessman Oskar Schind...,44.761,1993-12-15,8.6,12959
4,240,The Godfather: Part II,"Drama,Crime",en,In the continuing saga of the Corleone crime f...,57.749,1974-12-20,8.6,9811


## 2) Build text features

In [ ]:
movies_df['combined_text'] = (
    movies_df['title'].fillna('') + ' ' +
    movies_df['genre'].fillna('') + ' ' +
    movies_df['original_language'].fillna('') + ' ' +
    movies_df['overview'].fillna('')
)
movies_df[['title', 'combined_text']].head()

,title,combined_text
0,The Shawshank Redemption,"The Shawshank Redemption Drama,Crime en Framed..."
1,Dilwale Dulhania Le Jayenge,"Dilwale Dulhania Le Jayenge Comedy,Drama,Roman..."
2,The Godfather,"The Godfather Drama,Crime en Spanning the year..."
3,Schindler's List,"Schindler's List Drama,History,War en The true..."
4,The Godfather: Part II,"The Godfather: Part II Drama,Crime en In the c..."


## 3) Train similarity model

In [ ]:
vectorizer = TfidfVectorizer(stop_words='english', ngram_range=(1, 2), min_df=2)
text_matrix = vectorizer.fit_transform(movies_df['combined_text'])
similarity = cosine_similarity(text_matrix)

print('Matrix shape:', text_matrix.shape)
print('Similarity shape:', similarity.shape)

Matrix shape: (10000, 35108)
Similarity shape: (10000, 10000)


In [ ]:
def recommend_movie(movie_title: str, top_n: int = 5):
    title = movie_title.strip()
    if not title:
        return []

    match = movies_df[movies_df['title'].str.lower() == title.lower()]
    if match.empty:
        return []

    movie_index = match.index[0]
    scores = sorted(enumerate(similarity[movie_index]), key=lambda x: x[1], reverse=True)

    return [
        {
            'title': movies_df.iloc[idx]['title'],
            'genre': movies_df.iloc[idx]['genre'],
            'language': movies_df.iloc[idx]['original_language'],
            'score': float(score),
        }
        for idx, score in scores[1:top_n + 1]
    ]

recommend_movie('The Shawshank Redemption', 5)

[{'title': 'Sherlock Jr.',
  'genre': 'Action,Comedy,Mystery',
  'language': 'en',
  'score': 0.16957333950359296},
 {'title': 'Double Jeopardy',
  'genre': 'Thriller',
  'language': 'en',
  'score': 0.11903311001488077},
 {'title': 'The Blues Brothers',
  'genre': 'Music,Comedy,Action,Crime',
  'language': 'en',
  'score': 0.11260262590410061},
 {'title': 'Sleuth',
  'genre': 'Mystery,Thriller,Crime',
  'language': 'en',
  'score': 0.11066947347904349},
 {'title': 'Brubaker',
  'genre': 'Crime,Drama',
  'language': 'en',
  'score': 0.10901940615593562}]

## 4) Save model files

In [ ]:
movies_path = model_dir / 'movies.pkl'
similarity_path = model_dir / 'similarity.pkl'

with movies_path.open('wb') as f:
    pickle.dump(movies_df, f)

with similarity_path.open('wb') as f:
    pickle.dump(similarity, f)

print(f'Saved: {movies_path} and {similarity_path}')

Saved: movies.pkl and similarity.pkl
